# Pillar 1: DDI prediction — Colab training

Train the strict-identity DDI pilot in Colab using the prepared files in Google Drive. TWOSIDES supplies the pair labels; DDInter-known pairs are excluded from sampled negatives. The pilot compares Morgan logistic regression, edge-aware GATv2, and the proposed seven-input model. Model selection and post-hoc calibration use pair-disjoint S1-dev subsets; the frozen S1-test set remains untouched.

Run the cells in order. The pilot uses one seed (42); the neural models are configured for up to 200 epochs. Do not treat metrics as final until the run split manifest is compared with the fixed split audit in Drive.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# Find the Drive location already mounted in this Colab runtime.
data_candidates = [
    Path("/content/gdrive_auditddi_transfer/MyDrive/Projects/auditddi"),
    Path("/content/gdrive_auditddi/MyDrive/Projects/auditddi"),
    Path("/content/drive/MyDrive/Projects/auditddi"),
]
DATA = next((p for p in data_candidates if p.is_dir()), None)
if DATA is None:
    from google.colab import drive
    MOUNT = Path("/content/gdrive_auditddi_training")
    if MOUNT.exists() and any(MOUNT.iterdir()):
        MOUNT = Path("/content/gdrive_auditddi_training2")
    drive.mount(str(MOUNT))
    DATA = MOUNT / "MyDrive/Projects/auditddi"
assert DATA.is_dir(), f"Google Drive data root not found: {DATA}"

# Pull the current code from the GitHub repository. Publish the intended DDI code first.
REPO = Path("/content/auditddi_colab_training")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/akhilau-git/auditddi.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), f"Not a Git checkout: {REPO}"
    dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
    assert not dirty, "The Colab code checkout has local edits. Use a fresh runtime or review the checkout before pulling."
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
assert (REPO / "requirements_colab.txt").is_file(), "GitHub checkout is missing the Colab requirements file."

RESULTS_BASE = DATA / "results"
RUNS = RESULTS_BASE / "pillars" / "DDI"
RUNS.mkdir(parents=True, exist_ok=True)
os.environ["AUDITDDI_DATA_BASE"] = str(DATA)
os.environ["AUDITDDI_RESULTS_BASE"] = str(RESULTS_BASE)
os.environ["AUDITDDI_CODE_ROOT"] = str(REPO)
sys.path.insert(0, str(REPO / "src"))
print("Code checkout:", REPO)
print("Code commit:", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip())
print("Drive data:", DATA)
print("DDI results:", RUNS)


In [ ]:
%pip -q install -r {REPO / "requirements_colab.txt"}


In [ ]:
# Confirm GPU and all prepared inputs before training.
import torch
import py_compile
for script in [REPO / "src/training/train_full_pipeline_v2.py", REPO / "src/training/train_ecfp_logistic_baseline.py"]:
    py_compile.compile(str(script), doraise=True)
print("Training script syntax: OK")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "In Colab, choose Runtime > Change runtime type > a GPU accelerator, then rerun."

FEATURE_BUNDLE = DATA / "processed/auditddi_v3/seven_branch_features.npz"
PRETRAINED_ENCODER = DATA / "checkpoints/chembl_pretrained_encoder.pt"
FIXED_SPLITS = DATA / "processed/ddi_pillar/fixed_splits/20261026T090152_368722"
required = [
    REPO / "pillars/ddi/scripts/run_screening.py",
    REPO / "src/models/seven_branch_ddi.py",
    REPO / "src/data_prep/frozen_pair_splits.py",
    REPO / "src/training/run_experiment_suite.py",
    DATA / "processed/auditddi_v1/twosides_drug_cid_resolution.csv",
    DATA / "processed/auditddi_v1/ddinter_pubchem_match_audit.csv",
    DATA / "processed/auditddi_v2/twosides_pairs.csv",
    DATA / "processed/auditddi_v2/ddinter_pairs.csv",
    DATA / "processed/twosides_v1/twosides_clean.parquet",
    DATA / "processed/twosides_v1/drugs_clean.csv",
    FEATURE_BUNDLE, FEATURE_BUNDLE.with_suffix(".manifest.json"),
    PRETRAINED_ENCODER,
    FIXED_SPLITS / "strict_identity", FIXED_SPLITS / "include_provisional",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing DDI training inputs or code:\n" + "\n".join(missing)
print("All required DDI inputs and code are present.")
print("Feature bundle:", FEATURE_BUNDLE)
print("ChEMBL pretrained encoder:", PRETRAINED_ENCODER)
print("Audited split reference:", FIXED_SPLITS)
for cohort in ["strict_identity", "include_provisional"]:
    folder = FIXED_SPLITS / cohort
    print(f"{cohort}:", [p.name for p in sorted(folder.iterdir()) if p.is_file()])
print("Training loads frozen audited split CSVs directly and validates cohort, seed, identity exclusions, source hashes, row counts, and pair uniqueness.")


In [ ]:
# Run only the strict-identity pilot first.
from datetime import datetime
import pandas as pd

resolution = pd.read_csv(DATA / "processed/auditddi_v1/twosides_drug_cid_resolution.csv")
strict_excluded = sorted(resolution.loc[~resolution["match_status"].isin(["matched_rxnorm_and_exact_structure", "conflict_resolved_by_exact_structure"]), "drug_idx"].astype(int).tolist())
run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
env = os.environ.copy()
env.update({
    "AUDITDDI_EXPERIMENT_PRESET": "paper",
    "AUDITDDI_EXPERIMENT_EPOCHS": "200",
    "AUDITDDI_EXPERIMENT_SEEDS": "42",
    "AUDITDDI_EXPERIMENT_SPLIT_SEED": "42",
    "AUDITDDI_MODEL_SELECTION_SPLIT": "s1_dev",
    "AUDITDDI_RUN_PAIR_NOVELTY_AUDIT": "false",
    "AUDITDDI_BOOTSTRAP_RESAMPLES": "200",
    "AUDITDDI_EXPERIMENT_NAMES": "ecfp_sgd_logistic,edge_aware_ddi_only,seven_branch_ddi_only",
    "AUDITDDI_EXPERIMENT_REFERENCE": "ecfp_sgd_logistic",
    "AUDITDDI_SEVEN_BRANCH_FEATURE_BUNDLE": str(FEATURE_BUNDLE),
    "AUDITDDI_FIXED_PAIR_SPLITS_DIR": str(FIXED_SPLITS / "strict_identity"),
    "AUDITDDI_CHEMBL_PRETRAINED_ENCODER_PATH": str(PRETRAINED_ENCODER),
    "AUDITDDI_EXPERIMENTS_BASE": str(RUNS / "experiments" / "strict_identity"),
    "AUDITDDI_STUDY_ID": f"strict_identity_pilot_{run_tag}",
    "AUDITDDI_EXCLUDED_TWOSIDES_DRUG_IDS": ",".join(map(str, strict_excluded)),
    "PYTHONUNBUFFERED": "1",
})
env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")
print("Starting strict-identity pilot: seed 42; up to 200 epochs for neural models.")
subprocess.run([sys.executable, "-u", str(REPO / "pillars/ddi/scripts/run_screening.py")], cwd=REPO, env=env, check=True)
print("Pilot outputs:", RUNS / "experiments" / "strict_identity")
